## 섹션 0-1: 환경 감지 및 경로 설정

1. `git clone https://github.com/ehdgus4173/DACON_Structure_Stability.git`
2. 레포 상위 폴더에 `dataset/` 폴더 생성 후 데이콘 데이터 배치:
   ```
   DACON_Structure_Stability/  ← 레포
   dataset/                    ← 데이터 (레포와 같은 위치)
     train/ dev/ test/
     train.csv  dev.csv  sample_submission.csv
   ```
3. conda 환경 생성: `conda create -n stability python=3.10`
4. 패키지 설치: `pip install -r requirements.txt`
5. Jupyter 커널 등록: `python -m ipykernel install --user --name stability`
6. 피처 추출: `python src/features/extract_base.py && python src/features/extract_advanced.py`
7. 이 노트북 실행

In [ ]:
import os, sys
from pathlib import Path

IS_KAGGLE = os.path.exists('/kaggle')

if IS_KAGGLE:
    DATA_DIR    = '/kaggle/input/datasets/junseopkim/structure-stability-data/data'
    SRC_DIR     = '/kaggle/input/datasets/junseopkim/structure-stability-src/src'
    OUT_DIR     = '/kaggle/working/outputs'
    FEAT_DIR    = '/kaggle/working/features'
    IMG_SIZE    = 384
    NUM_WORKERS = 2
else:
    REPO_ROOT = Path('..').resolve()
    sys.path.insert(0, str(REPO_ROOT))
    from config import DATASET_DIR, PROJECT_ROOT, PHYS_COLS_V2
    DATA_DIR    = str(DATASET_DIR)
    SRC_DIR     = str(REPO_ROOT / 'src')
    OUT_DIR     = str(REPO_ROOT / 'outputs')
    FEAT_DIR    = str(REPO_ROOT / 'features')
    IMG_SIZE    = 224
    NUM_WORKERS = 0

sys.path.append(SRC_DIR)
FEATURES_CSV = os.path.join(FEAT_DIR, 'combined_features_v3.csv')

FIG_DIR = f"{OUT_DIR}/../reports/figures" if not IS_KAGGLE else f"{OUT_DIR}/figures"
os.makedirs(f'{OUT_DIR}/models',      exist_ok=True)
os.makedirs(f'{OUT_DIR}/logs',        exist_ok=True)
os.makedirs(f'{OUT_DIR}/submissions', exist_ok=True)
os.makedirs(FEAT_DIR,                 exist_ok=True)
os.makedirs(FIG_DIR,                  exist_ok=True)

print(f"환경: {'Kaggle' if IS_KAGGLE else 'Local'}")
print(f"DATA_DIR: {DATA_DIR}")
print(f"FEATURES_CSV: {FEATURES_CSV}")
print(f"IMG_SIZE: {IMG_SIZE}")
print(f"GPU: {os.popen('nvidia-smi --query-gpu=name --format=csv,noheader').read().strip()}")

## 섹션 0-1b: 물리 피처 추출 (최초 1회만 실행)
combined_features_v3.csv가 없을 때만 extract_base.py → extract_advanced.py 순서로 실행합니다.

In [ ]:
import subprocess

if not os.path.exists(FEATURES_CSV):
    print('Feature file not found - extracting (5~10 min)...')
    env = os.environ.copy()
    if IS_KAGGLE:
        env['KAGGLE_MODE']  = '1'
        env['DATASET_DIR']  = DATA_DIR
        env['FEAT_OUT_DIR'] = FEAT_DIR
    r1 = subprocess.run(
        [sys.executable, os.path.join(SRC_DIR, 'features', 'extract_base.py')],
        capture_output=True, text=True, env=env
    )
    print(r1.stdout[-800:] if r1.stdout else '')
    if r1.returncode != 0:
        print('extract_base error:', r1.stderr[-500:])
    r2 = subprocess.run(
        [sys.executable, os.path.join(SRC_DIR, 'features', 'extract_advanced.py')],
        capture_output=True, text=True, env=env
    )
    print(r2.stdout[-800:] if r2.stdout else '')
    if r2.returncode != 0:
        print('extract_advanced error:', r2.stderr[-500:])
    print(f'Done: {FEATURES_CSV}')
else:
    print(f'Already exists: {FEATURES_CSV}')

import pandas as pd
feat_df = pd.read_csv(FEATURES_CSV)
print(f'Feature shape: {feat_df.shape}')


## 섹션 0-2: Import 및 DEVICE 설정


In [ ]:
import importlib, sys

# 세션 재시작 없이도 최신 버전 강제 로드
for mod_name in ['experiment_utils', 'dataset', 'model', 'augmentation']:
    if mod_name in sys.modules:
        importlib.reload(sys.modules[mod_name])

from model import MultiViewNet
from experiment_utils import set_seed, EarlyStopping, run_experiment, run_inference, train_one_epoch, evaluate

import random, json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
from sklearn.metrics import log_loss, accuracy_score

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")

## 섹션 1-2: 모델 구조 확인


In [ ]:
test_model = MultiViewNet(backbone_name='resnet18', fusion_mode='concat', img_size=IMG_SIZE).to(DEVICE)
dummy = torch.randn(2, 3, IMG_SIZE, IMG_SIZE).to(DEVICE)
out = test_model(dummy, dummy)
print(f"출력 shape: {out.shape}")
print(f"총 파라미터: {sum(p.numel() for p in test_model.parameters()):,}")
del test_model, dummy

## 섹션 1-3: 베이스 Config 사전 정의 (EXP-017~020 단독 실행 시 필수)
Run this cell first before EXP-017~020.
Execution order: 0-1 -> 0-1b -> 0-2 -> 1-2 -> **1-3** -> then jump to EXP-017~020

In [ ]:
# Base Config Pre-definition
# Run this cell to define config_exp008 and config_exp012
# without re-running all previous experiment cells

config_exp008 = {
    "exp_id": "008", "model_version": "8", "model_name": "resnet18",
    "backbone": "resnet18", "fusion_mode": "diff_concat",
    "use_physics": True, "physics_dim": 6,
    "features_csv": FEATURES_CSV,
    "data_dir": DATA_DIR, "train_csv": "train.csv", "dev_csv": "dev.csv",
    "submission_csv": "sample_submission.csv",
    "norm_version": "custom", "dev_split_ratio": 0.8, "img_size": IMG_SIZE,
    "epochs": 50, "lr": 1e-3, "batch_size": 16,
    "early_stopping_patience": 7, "num_workers": NUM_WORKERS,
    "random_state": 42, "out_dir": OUT_DIR
}

config_exp012 = {
    **config_exp008,
    "exp_id": "012", "model_version": "12",
    "dev_split_ratio": 0.5,
    "early_stopping_patience": 10
}

print("config_exp008 / config_exp012 ready")
print(f"  features_csv : {FEATURES_CSV}")
print(f"  data_dir     : {DATA_DIR}")
print(f"  img_size     : {IMG_SIZE}")


## 섹션 2: EXP-001 — HIGH 증강 (Ablation)


In [ ]:
config_exp001 = {
    "exp_id": "001", "model_version": "2", "model_name": "resnet18",
    "backbone": "resnet18", "fusion_mode": "concat",
    "use_physics": False, "physics_dim": 6,
    "features_csv": FEATURES_CSV,
    "data_dir": DATA_DIR, "train_csv": "train.csv", "dev_csv": "dev.csv",
    "submission_csv": "sample_submission.csv",
    "norm_version": "imagenet", "dev_split_ratio": 0.0, "img_size": IMG_SIZE,
    "epochs": 50, "lr": 1e-3, "batch_size": 16,
    "early_stopping_patience": 5, "num_workers": NUM_WORKERS,
    "random_state": 42, "out_dir": OUT_DIR
}
result_exp001 = run_experiment(config_exp001)

## 섹션 3: EXP-002 — Dev 학습 포함 (Ablation)


In [ ]:
config_exp002 = {**config_exp001, "exp_id": "002", "model_version": "3", "dev_split_ratio": 0.8}
result_exp002 = run_experiment(config_exp002)

## 섹션 4: EXP-004 — Custom 정규화 (Ablation)


In [ ]:
config_exp004 = {**config_exp001, "exp_id": "004", "model_version": "4", "norm_version": "custom"}
result_exp004 = run_experiment(config_exp004)

## 섹션 5: EXP-005 — EfficientNet-B0 백본 (Ablation)


In [ ]:
config_exp005 = {**config_exp001, "exp_id": "005", "model_version": "5",
                 "model_name": "efficientnet", "backbone": "efficientnet_b0"}
result_exp005 = run_experiment(config_exp005)

## 섹션 6: EXP-006 — 물리 피처 6개 (Ablation)


In [ ]:
config_exp006 = {**config_exp001, "exp_id": "006", "model_version": "6",
                 "use_physics": True, "physics_dim": 6}
result_exp006 = run_experiment(config_exp006)

## 섹션 7: EXP-007 — diff_concat Fusion (Ablation)


In [ ]:
config_exp007 = {**config_exp001, "exp_id": "007", "model_version": "7", "fusion_mode": "diff_concat"}
result_exp007 = run_experiment(config_exp007)

## 섹션 8: EXP-008 — 최적 조합 ✅ Public 0.16137 / 175위
Custom 정규화 + diff_concat Fusion + Dev 포함 + 물리 피처 6개 + HIGH 증강
결과: Best Val LogLoss 0.1028 (Epoch 2) — Public 0.16137 / 175위

In [ ]:
config_exp008 = {
    "exp_id": "008", "model_version": "8", "model_name": "resnet18",
    "backbone": "resnet18", "fusion_mode": "diff_concat",
    "use_physics": True, "physics_dim": 6,
    "features_csv": FEATURES_CSV,
    "data_dir": DATA_DIR, "train_csv": "train.csv", "dev_csv": "dev.csv",
    "submission_csv": "sample_submission.csv",
    "norm_version": "custom", "dev_split_ratio": 0.8, "img_size": IMG_SIZE,
    "epochs": 50, "lr": 1e-3, "batch_size": 16,
    "early_stopping_patience": 7, "num_workers": NUM_WORKERS,
    "random_state": 42, "out_dir": OUT_DIR
}
result_exp008 = run_experiment(config_exp008)

## 섹션 9: EXP-009~011 — dev_split_ratio Ablation (0.0 / 0.5 / 1.0)


In [ ]:
for ratio, exp_id, ver in [(0.0, '009', '9'), (0.5, '010', '10'), (1.0, '011', '11')]:
    cfg = {
        **config_exp008,
        "exp_id": exp_id, "model_version": ver,
        "dev_split_ratio": ratio,
        "epochs": 3 if ratio == 1.0 else 50,
        "early_stopping_patience": 7
    }
    result = run_experiment(cfg)
    print(f"ratio={ratio} → Best Val LogLoss: {result['best_val_logloss']:.4f}")

## 섹션 10: EXP-012 — 최적 조합 + ratio=0.5 ✅ Val 0.0608
결과: Best Val LogLoss 0.0608 (Epoch 19)

In [ ]:
config_exp012 = {
    **config_exp008,
    "exp_id": "012", "model_version": "12",
    "dev_split_ratio": 0.5,
    "early_stopping_patience": 10
}
result_exp012 = run_experiment(config_exp012)

## 섹션 11: 추론 — submission_v2.csv (EXP-008 기준)
Public LogLoss: 0.16137 / 175위


In [ ]:
preds_v2 = run_inference(config=config_exp008, model_path=result_exp008['model_path'])
sub_df = pd.read_csv(f"{DATA_DIR}/sample_submission.csv")
sub_df['unstable_prob'] = preds_v2
sub_df['stable_prob']   = 1.0 - preds_v2
sub_df.to_csv(f"{OUT_DIR}/submissions/submission_v2.csv", index=False)
print(f"저장 완료 | prob 범위: {preds_v2.min():.4f}~{preds_v2.max():.4f}")

## 섹션 12: EXP-013 — CosineAnnealingLR
결과: Best Val LogLoss 0.0756 (Epoch 13)

In [ ]:
config_exp013 = {
    **config_exp012,
    "exp_id": "013", "model_version": "13",
    "lr_scheduler": "cosine", "lr_min": 1e-5,
    "epochs": 50, "early_stopping_patience": 10
}
result_exp013 = run_experiment(config_exp013)

## 섹션 13: EXP-014 — EfficientNet-B4 백본
결과: Best Val LogLoss 0.3281 (Epoch 1)

In [ ]:
config_exp014 = {
    **config_exp012,
    "exp_id": "014", "model_version": "14",
    "model_name": "efficientnet_b4", "backbone": "efficientnet_b4",
    "batch_size": 8, "lr_scheduler": "cosine", "lr_min": 1e-5,
    "epochs": 50, "early_stopping_patience": 10
}
result_exp014 = run_experiment(config_exp014)

## 섹션 14: EXP-015 — Seed 앙상블 (42/43/44) ✅ Val 0.0438
결과: seed44=0.0438 (전체 최고) — Public 0.07936 / 100위

In [ ]:
seed_preds = []
for seed in [42, 43, 44]:
    cfg = {
        **config_exp012,
        "exp_id": f"015_seed{seed}", "model_version": f"15s{seed}",
        "random_state": seed, "epochs": 50, "early_stopping_patience": 10
    }
    result = run_experiment(cfg)
    preds  = run_inference(config=cfg, model_path=result['model_path'])
    seed_preds.append(preds)
    print(f"seed={seed} → Best Val LogLoss: {result['best_val_logloss']:.4f}")

ensemble_preds = np.mean(seed_preds, axis=0)
sub_df = pd.read_csv(f"{DATA_DIR}/sample_submission.csv")
sub_df['unstable_prob'] = ensemble_preds
sub_df['stable_prob']   = 1.0 - ensemble_preds
sub_df.to_csv(f"{OUT_DIR}/submissions/submission_v3_ensemble.csv", index=False)
print(f"앙상블 완료 | prob 범위: {ensemble_preds.min():.4f}~{ensemble_preds.max():.4f}")

## 섹션 15: EXP-016 — Phys MLP 구조
결과: Best Val LogLoss 0.1328 (Epoch 5)

In [ ]:
config_exp016 = {
    **config_exp012,
    "exp_id": "016", "model_version": "16",
    "use_phys_mlp": True, "epochs": 50, "early_stopping_patience": 10
}
result_exp016 = run_experiment(config_exp016)

## 섹션 16: 추론 — submission_v3.csv (EXP-012 기준)


In [ ]:
preds_v3 = run_inference(config=config_exp012, model_path=result_exp012['model_path'])
sub_df = pd.read_csv(f"{DATA_DIR}/sample_submission.csv")
sub_df['unstable_prob'] = preds_v3
sub_df['stable_prob']   = 1.0 - preds_v3
sub_df.to_csv(f"{OUT_DIR}/submissions/submission_v3.csv", index=False)
print(f"저장 완료 | prob 범위: {preds_v3.min():.4f}~{preds_v3.max():.4f}")

## 섹션 17: EXP-017 — 팀원 피처 20개 (physics_dim 6→20)
physics_dim 6 → 20 (팀원 전체 피처)

In [ ]:
config_exp017 = {
    **config_exp012,
    "exp_id": "017", "model_version": "17",
    "use_physics": True, "physics_dim": 20,
    "epochs": 50, "early_stopping_patience": 10
}
result_exp017 = run_experiment(config_exp017)

## 섹션 18: EXP-018 — 팀원 피처 20개 + Phys MLP


In [ ]:
config_exp018 = {
    **config_exp017,
    "exp_id": "018", "model_version": "18",
    "use_phys_mlp": True,
    "epochs": 50, "early_stopping_patience": 10
}
result_exp018 = run_experiment(config_exp018)

## 섹션 19: EXP-019 — 격자 피처 Ablation (physics_dim=14)


In [ ]:
config_exp019 = {
    **config_exp012,
    "exp_id": "019", "model_version": "19",
    "use_physics": True, "physics_dim": 14,
    "epochs": 50, "early_stopping_patience": 10
}
result_exp019 = run_experiment(config_exp019)

## 섹션 20: EXP-020 — Seed 앙상블 (팀원 피처 최적 config) → submission_v5.csv
EXP-017 vs EXP-018 결과 보고 더 좋은 config로 변경 후 실행


In [ ]:
# EXP-017 vs EXP-018 결과 보고 더 좋은 것으로 변경
best_phys_config = config_exp017

seed_preds_v5 = []
for seed in [42, 43, 44]:
    cfg = {
        **best_phys_config,
        "exp_id": f"020_seed{seed}", "model_version": f"20s{seed}",
        "random_state": seed, "epochs": 50, "early_stopping_patience": 10
    }
    result = run_experiment(cfg)
    preds  = run_inference(config=cfg, model_path=result['model_path'])
    seed_preds_v5.append(preds)
    print(f"seed={seed} → Best Val LogLoss: {result['best_val_logloss']:.4f}")

ensemble_v5 = np.mean(seed_preds_v5, axis=0)
sub_df = pd.read_csv(f"{DATA_DIR}/sample_submission.csv")
sub_df['unstable_prob'] = np.clip(ensemble_v5, 1e-7, 1-1e-7)
sub_df['stable_prob']   = 1.0 - sub_df['unstable_prob']
sub_df.to_csv(f"{OUT_DIR}/submissions/submission_v5.csv", index=False)
print(f"submission_v5.csv 저장 완료 | prob 범위: {ensemble_v5.min():.4f}~{ensemble_v5.max():.4f}")

## 섹션 21: 전체 실험 결과 비교표


In [ ]:
results_all = [
    {"exp_id": "baseline", "변경사항": "없음(기준점)",      "best_val_logloss": 1.5369, "public": 1.67225},
    {"exp_id": "001",      "변경사항": "HIGH 증강",          "best_val_logloss": 0.5007, "public": None},
    {"exp_id": "002",      "변경사항": "Dev 포함(0.8)",      "best_val_logloss": 0.3687, "public": None},
    {"exp_id": "004",      "변경사항": "Custom 정규화",       "best_val_logloss": 0.1700, "public": None},
    {"exp_id": "005",      "변경사항": "EfficientNet-B0",    "best_val_logloss": 0.5081, "public": None},
    {"exp_id": "006",      "변경사항": "물리 피처 6개",       "best_val_logloss": 0.3438, "public": None},
    {"exp_id": "007",      "변경사항": "diff_concat Fusion", "best_val_logloss": 0.1996, "public": None},
    {"exp_id": "008",      "변경사항": "최적 조합",           "best_val_logloss": 0.1028, "public": 0.16137},
    {"exp_id": "009",      "변경사항": "ratio=0.0",          "best_val_logloss": 0.1880, "public": None},
    {"exp_id": "010",      "변경사항": "ratio=0.5",          "best_val_logloss": 0.1037, "public": None},
    {"exp_id": "011",      "변경사항": "ratio=1.0",          "best_val_logloss": 0.5896, "public": None},
    {"exp_id": "012",      "변경사항": "최적조합+ratio=0.5", "best_val_logloss": 0.0608, "public": None},
    {"exp_id": "013",      "변경사항": "CosineAnnealingLR", "best_val_logloss": 0.0756, "public": None},
    {"exp_id": "014",      "변경사항": "EfficientNet-B4",   "best_val_logloss": 0.3281, "public": None},
    {"exp_id": "015_s42",  "변경사항": "Seed42",            "best_val_logloss": 0.0994, "public": None},
    {"exp_id": "015_s43",  "변경사항": "Seed43",            "best_val_logloss": 0.0491, "public": None},
    {"exp_id": "015_s44",  "변경사항": "Seed44",            "best_val_logloss": 0.0438, "public": None},
    {"exp_id": "016",      "변경사항": "Phys MLP",          "best_val_logloss": 0.1328, "public": None},
    {"exp_id": "v4",       "변경사항": "앙상블+TempScale",   "best_val_logloss": None,   "public": 0.07936},
]
result_df = pd.DataFrame(results_all).sort_values('best_val_logloss')
print(result_df.to_string(index=False))
result_df.to_csv(f"{OUT_DIR}/logs/experiment_summary.csv", index=False)